# Phase 4 CJPE — 03 C1 Full

One independent T4 notebook. All source is embedded. This notebook can start without any other notebook finishing. 

Required own checkpoint bundles: **Phase3_C1_Input.zip**. The Phase3 filename is historical: these contain original Phase1 LM weights, matching tokenizers and manifests. No Phase2 NER checkpoint is needed.

## Launch this notebook in its own account/runtime
1. In that account's Drive create **My Drive / CS781_Phase4 / inputs**.
2. Upload these existing Windows files there, without extracting:
- `D:\CS781\Phase4_20261020\inputs\Phase3_C1_Input.zip`
3. Select **T4 GPU**. Run the setup cells in order, approve Drive using the same Google account, and sign into your own Hugging Face account with course-dataset access when requested. Finish authentication before leaving it running.
4. Run CONFIG and **Start/resume**. That cell checks GPU gradients, prepares LOCAL SQLite caches, trains, validates, copies the submission ZIP into **My Drive / CS781_Phase4 / SUBMISSIONS**, then packages model/source records. No separate status/packaging cell is required.
5. Confirm status reaches **training** with increasing optimizer steps. Indexing/tokenizing and a preflight with optimizer_steps0 are preparation stages.

All three observed fixes are embedded: identical train/dev copies counted once, original Unicode13 tokenizer compatibility, and SQLite caches on local runtime disk. Do not add any old repair cells to this fresh notebook.

Select **Runtime → Change runtime type → T4 GPU**. Use only corrected official CJPE data. Login/mount approvals need your action before leaving it running. Actual GPU time, scores and schema compatibility have not been established in the authoring environment.

Persistent checkpoints/results: `MyDrive/CS781_Phase4/runs/03_C1_Full/`. Every result ZIP contains only root prediction.json. MODEL_RECORDS ZIPs are for your records and must not be submitted to Codabench.

Deadline: **21 October 2026, 23:59 IST**. Default training stops early enough to reserve 45 minutes for export. An early result is exported after the first holdout evaluation; use the final result when it is ready. No automated Codabench uploads.


In [ ]:
# Keep Colab's CUDA-enabled torch and existing tokenizers; do not install FlashAttention or model weights.
%pip -q install "ijson>=3.2,<4" "huggingface_hub>=0.26,<2" "scikit-learn>=1.4,<2"
import importlib.util,subprocess,sys
if importlib.util.find_spec('tokenizers') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers'])
import torch,platform,pathlib,shutil
print('Python:',platform.python_version(),'Torch:',torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print('ACTUAL GPU:',torch.cuda.get_device_name(0))
print('CUDA:',torch.version.cuda,'free local GB:',round(shutil.disk_usage('/').free/1e9,2))


In [ ]:
# Embedded source with exact SHA verification; no separate source upload.
import base64,hashlib,io,zipfile,pathlib,sys
source_bytes=base64.b64decode('...')
assert hashlib.sha256(source_bytes).hexdigest()=='0d776ceb1277195ad3c1f59b2c0a83d3b0d180b6ac3641d3196547e0ab309dbb'
CODE_ROOT=pathlib.Path('/content/phase4_code' if pathlib.Path('/content').exists() else '/kaggle/working/phase4_code')
CODE_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
    for item in z.infolist():
        p=pathlib.PurePosixPath(item.filename)
        assert not p.is_absolute() and '..' not in p.parts
        dest=CODE_ROOT/item.filename;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(item))
if str(CODE_ROOT) not in sys.path:sys.path.insert(0,str(CODE_ROOT))
print('Source ready:',CODE_ROOT)


In [ ]:
# Choose how your own checkpoint bundles are supplied.
from pathlib import Path
INPUT_MODE='drive'  # 'drive' or 'temporary_upload'
if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST=Path('/content/drive/MyDrive/CS781_Phase4')
    WORK=Path('/content/phase4_work')
else:
    PERSIST=Path('/kaggle/working/CS781_Phase4');WORK=PERSIST/'work'
PERSIST.mkdir(parents=True,exist_ok=True);WORK.mkdir(parents=True,exist_ok=True)
INPUT_DIR=PERSIST/'inputs' if INPUT_MODE=='drive' else Path('/content/inputs')
# For an already-uploaded Phase3 bundle you may set INPUT_DIR to that Drive folder.
REQUIRED=['Phase3_C1_Input.zip']
EXPECTED={'Phase3_C1_Input.zip': {'bytes': 1414226371, 'sha256': '79cb787cd1efdfd9248adf5a74a315de5f38edcd20882b54077b7bcedfcc7d90'}}
from phase4_cjpe.common import sha
for filename in REQUIRED:
    path=INPUT_DIR/filename
    # Optional support for the exact numbered parts already uploaded in Phase3.
    if not path.exists():
        parts=sorted(INPUT_DIR.glob(filename+'.part*'))
        if parts:
            assert [p.name for p in parts]==[filename+f'.part{i:03d}' for i in range(1,len(parts)+1)], 'Incomplete numbered input parts'
            path=WORK/filename
            with path.with_suffix('.partial').open('wb') as out:
                for part in parts:
                    with part.open('rb') as src:shutil.copyfileobj(src,out,8*1024*1024)
            path.with_suffix('.partial').replace(path)
    assert path.exists(), f'Upload {filename} to {INPUT_DIR} using Drive, or the Colab Files pane for temporary mode.'
    assert path.stat().st_size==EXPECTED[filename]['bytes'], 'Incomplete checkpoint upload'
    assert sha(path)==EXPECTED[filename]['sha256'], 'Checkpoint bundle SHA mismatch'
    print('Verified input:',path)
    EXPECTED[filename]['resolved_path']=str(path)


In [ ]:
# Official corrected data: uploaded ZIP, existing folder, or explicit approved HF login.
from phase4_cjpe.data import acquire,index_data
DATA_ROOT=WORK/'Phase_4_CJPE'
OFFICIAL_DATA_ZIP=None  # Example: str(INPUT_DIR/'Phase_4_CJPE.zip')
# Alternatively upload Train/train.json, Train/dev.json, Test/test.json to DATA_ROOT.
missing=any(not (DATA_ROOT/relative).exists() for relative in ('Train/train.json','Train/dev.json','Test/test.json'))
if missing and not OFFICIAL_DATA_ZIP:
    from huggingface_hub import notebook_login
    notebook_login()  # Authenticate only to your own account with approved official-dataset access.
DATA_ROOT=acquire(DATA_ROOT,uploaded_zip=OFFICIAL_DATA_ZIP,download=bool(missing and not OFFICIAL_DATA_ZIP))
print('Official data ready:',DATA_ROOT)
print('The worker will stream records and check labels/IDs and cross-split text duplication before training.')


In [ ]:
# Configure BEFORE first run; keep it unchanged when resuming the same RUN_ROOT.
CONFIG={'name': '03_C1_Full', 'family': 'c1', 'method': 'full', 'experiment': 'adaptation', 'rank': 8, 'alpha': 16, 'seed': 781, 'max_tokens': 256, 'chunks_per_case': 4, 'region_chars': 8192, 'max_train_cases': 12000, 'holdout_per_class': 256, 'max_epochs': 4, 'accumulation': 8, 'backbone_lr': 1e-05, 'head_lr': 0.0003, 'training_hours': 3.5, 'eval_every': 100, 'save_every': 25, 'patience': 5, 'teacher_hours': 1.5, 'kd_temperature': 2.0, 'kd_weight': 0.5, 'stall_minutes': 45, 'export_reserve_minutes': 45, 'deadline_utc': '2026-10-21T18:29:00Z', 'allow_after_deadline': False}
RUN_ROOT=PERSIST/'runs'/'03_C1_Full'
CONFIG.update(run_root=str(RUN_ROOT),work_root=str(WORK),data_root=str(DATA_ROOT),
              bundle=EXPECTED['Phase3_C1_Input.zip']['resolved_path'])

print(CONFIG)
print('12,000 is the initial TRAINING-CASE cap, not a token count. Use None for the full official train split in a new run.')
print('Nominal training budget:',CONFIG['training_hours'],'hours; teacher adds',CONFIG['teacher_hours'] if CONFIG['experiment']=='distillation' else 0)


In [ ]:
# START/RESUME: GPU preflight, training, submission export and model/source records.
import copy,gc,json,shutil,time
from pathlib import Path
import torch
from IPython.display import FileLink,display
from phase3_rr.archive_inputs import resolve_bundle
from phase3_rr.data import Tokenizer
from phase4_cjpe.common import atomic_json,sha,status,source_hashes,now
from phase4_cjpe.models import make_model
from phase4_cjpe.runtime import supervise
from phase4_cjpe.export import validate_zip,package_records

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
RUN_ROOT=Path(CONFIG['run_root']);RUN_ROOT.mkdir(parents=True,exist_ok=True)
for process in Path('/proc').iterdir():
    if not process.name.isdigit():continue
    try:command=process.joinpath('cmdline').read_bytes().split(b'\0')
    except (FileNotFoundError,PermissionError,ProcessLookupError):continue
    if b'phase4_cjpe.runtime' in command and b'--worker' in command:
        raise RuntimeError('A Phase4 worker is already active in this runtime. PID='+process.name)

def _preflight(cfg,stage):
    status(RUN_ROOT,'gpu_preflight',error=None,note=stage+'; optimizer training has not started.')
    bundle,manifest=resolve_bundle(cfg['bundle'],Path(cfg['work_root'])/'weights')
    assert manifest['family']==cfg['family'], 'Wrong input checkpoint family'
    original_sha=sha(bundle/'tokenizer.json')
    tok=Tokenizer(bundle/'tokenizer.json')
    metadata=json.loads((bundle/'tokenizer.json').read_text())
    text='The Supreme Court considers the appeal under section १२३.'
    tokens=tok.encode(text)
    assert tokens, 'Tokenizer produced no tokens'
    if metadata.get('format')=='cs781-own-byte-bpe-v1':
        assert tok.impl.decode(tokens)==text, 'Original own tokenizer round-trip failed'
    assert sha(bundle/'tokenizer.json')==original_sha, 'Original tokenizer changed'
    model=None
    try:
        model,evidence=make_model(bundle,cfg['family'],cfg['method'],cfg['rank'],cfg['alpha'])
        model=model.to('cuda').train()
        ids=[1]+tokens[:30]+[2]
        with torch.autocast(device_type='cuda',dtype=torch.float16):
            logits=model([ids],[0.0],torch.device('cuda')).float()
        loss=torch.nn.functional.cross_entropy(logits,torch.tensor([0],device='cuda'))
        assert bool(torch.isfinite(loss)), 'Nonfinite GPU preflight loss'
        scaler=torch.amp.GradScaler('cuda',init_scale=128)
        scaler.scale(loss).backward()
        grads=[p.grad for p in model.backbone.parameters() if p.requires_grad and p.grad is not None]
        assert grads and all(bool(torch.isfinite(g).all()) for g in grads), 'Nonfinite/missing backbone gradients'
        assert any(bool(g.abs().max()>0) for g in grads), 'No nonzero backbone/adaptation gradients'
        record={'stage':stage,'checked_utc':now(),'gpu':torch.cuda.get_device_name(0),
                'family':cfg['family'],'method':cfg['method'],'finite_loss':float(loss.detach()),
                'parameters':evidence['parameters'],'trainable_parameters':evidence['trainable_parameters'],
                'tokenizer_sha256':original_sha,'saved_unicode_version':metadata.get('unicode_version'),
                'parent_sha256':evidence['parent_sha256'],'nonzero_backbone_gradients':True,'optimizer_steps':0,
                'source':source_hashes()}
        atomic_json(RUN_ROOT/('preflight_'+stage+'_'+str(int(time.time()))+'.json'),record)
        print('GPU PREFLIGHT PASSED:',stage,{k:record[k] for k in ('gpu','family','method','parameters','trainable_parameters')},flush=True)
        return record
    finally:
        del model;gc.collect();torch.cuda.empty_cache()

if not (RUN_ROOT/'complete.json').exists():
    main_check=_preflight(CONFIG,'student' if CONFIG['experiment']=='distillation' else 'model')
    if CONFIG['experiment']=='distillation':
        teacher_cfg=copy.deepcopy(CONFIG)
        teacher_cfg.update(family='d222',method='lorma',bundle=CONFIG['teacher_bundle'])
        teacher_check=_preflight(teacher_cfg,'teacher')
        assert teacher_check['parameters']>main_check['parameters'], 'Teacher must be larger than the A0 student'
        print('Independent teacher/student preflights passed. Teacher training runs first.',flush=True)
revision=Path(CONFIG['data_root'])/'download_revision.json'
if revision.exists():shutil.copy2(revision,RUN_ROOT/'official_download_revision.json')
print('LOCAL SQLITE CACHE:',Path(CONFIG['work_root'])/'cjpe_cache_local'/CONFIG['name'],flush=True)
print('PERSISTENT CHECKPOINTS:',RUN_ROOT,flush=True)
status(RUN_ROOT,'starting_worker',error=None,note='GPU checks passed; preparing local caches before optimizer training.')
try:
    RESULT=supervise(CONFIG)
except Exception:
    supervisor=RUN_ROOT/'supervisor.json'
    active=Path(json.loads(supervisor.read_text()).get('active_config',CONFIG)['run_root']) if supervisor.exists() else RUN_ROOT
    failure=active/'failure.json'
    if failure.exists():print('CURRENT FAILURE:\n'+json.loads(failure.read_text()).get('traceback',''))
    raise
official_ids=json.loads((Path(RESULT['run_root'])/'splits.json').read_text())['test']
checks=validate_zip(RESULT['submission'],official_ids)
assert checks['sha256']==RESULT['sha256']
READY=Path('/content/drive/MyDrive/CS781_Phase4/SUBMISSIONS') if Path('/content/drive/MyDrive').exists() else Path(CONFIG['run_root']).parents[1]/'SUBMISSIONS'
READY.mkdir(parents=True,exist_ok=True)
SUBMISSION=READY/Path(RESULT['submission']).name
if SUBMISSION.resolve()!=Path(RESULT['submission']).resolve():shutil.copy2(RESULT['submission'],SUBMISSION)
assert sha(SUBMISSION)==RESULT['sha256']
atomic_json(READY/(SUBMISSION.stem+'_RESULT.json'),RESULT)
print('SUBMISSION ZIP READY:',SUBMISSION,flush=True)
print('LOCAL DEV MACRO F1:',RESULT['dev_metrics']['macro_f1'],flush=True)
print('COMPLETED AFTER DEADLINE:',RESULT['completed_after_deadline'],flush=True)
display(FileLink(str(SUBMISSION)))
# The prediction ZIP is already safe before optional archive creation starts.
try:
    RECORDS=package_records(RESULT)
    print('MODEL/SOURCE RECORDS READY:',RECORDS,flush=True)
except Exception as error:
    print('Prediction ZIP remains safe. Optional records archive failed:',repr(error),flush=True)


In [ ]:
# Current status + exact result link. Run this after completion or interruption.
import json
from IPython.display import display,FileLink
supervisor_file=RUN_ROOT/'supervisor.json'
active=Path(json.loads(supervisor_file.read_text())['active_config']['run_root']) if supervisor_file.exists() else RUN_ROOT
for filename in ('status.json','failure.json','latest_export.json','complete.json'):
    path=active/filename
    if path.exists():print(filename+(' (historical failure record; see status/complete for current state)' if filename=='failure.json' else ''),json.loads(path.read_text()))
export=active/'latest_export.json'
if export.exists():
    result=json.loads(export.read_text());print('SUBMISSION ZIP:',result['submission']);display(FileLink(result['submission']))
    print('Colab: use the Files pane / mounted Drive to download this exact ZIP. Choose D:/CS781/Phase4_20261020/deliverables on Windows.')


In [ ]:
# Optional model-records archive AFTER the prediction ZIP is safely saved.
# This can take extra minutes and hundreds of MB; enable only when ready.
MAKE_MODEL_RECORDS=False
if MAKE_MODEL_RECORDS:
    from phase4_cjpe.export import package_records
    records=package_records(RESULT)
    print('Records only; NOT for Codabench:',records)


## Resume and failure handling

Rerun with the same source, data, input ZIP and CONFIG. The full latest.pt restores optimizer/scaler/RNG and the data cursor; best.pt is the selected task model. Historical Phase1 optimizer buffers are not reused for this new task.

Status files contain real optimizer step and last-update time. A still-open tab is not training evidence. The supervisor retries transient I/O failures up to twice and moves OOM recovery to a separate output folder with smaller chunks and explicitly recorded fresh-optimizer warm start. It does not repair unknown code/data errors or resurrect a revoked Colab runtime. Login/Drive/GPU access must be established while you are present.

Only upload UPLOAD_Phase4_*.zip containing root prediction.json. Local macro-F1 is not a hidden leaderboard score. Preserve source, checkpoints and authentic failure logs for the report. Do not use old dev or leaked test labels during fine-tuning.

Before downloading, enable the browser's Ask where to save downloads and select your D: folder. This notebook does not silently change your Windows download settings.

References: official CS781 capstone and IL-TUR; LoRMA (Bihany et al.2025), adapters (Houlsby et al.2019), BitFit (Ben Zaken et al.2022), knowledge distillation (Hinton et al.2015). Own original source is preserved in the embedded package. Record actual hardware and coding assistance honestly.
